# Profilage du jeu de donnees DVF 2022 — Etape 1 : decouverte

Ce notebook realise un **premier profilage** du fichier `dvf-2022.parquet`
pour en decouvrir la structure, le volume, les types de donnees,
les valeurs manquantes et les premieres statistiques descriptives.

**Technique utilisee** : DuckDB, un moteur SQL leger qui interroge
le fichier Parquet directement sur le disque, sans charger l'ensemble
des donnees en memoire vive (RAM).

---

## Mode d'emploi

1. Le chemin du fichier est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).
3. Si DuckDB n'est pas installe, la cellule 1 s'en charge.

## Cellule 1 — Installation de DuckDB

A executer **une seule fois**. Si DuckDB est deja installe,
la cellule se termine sans rien faire.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("DuckDB pret.")

DuckDB pret.


## Cellule 2 — Reglages

**Seule cellule a modifier.** Indiquer le chemin complet vers le fichier
`.parquet` et le dossier ou enregistrer les resultats.

Le prefixe `r"..."` (raw string) evite les problemes d'antislash sous Windows.

In [2]:
import duckdb
import os
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

# Chemin a renseigner : dossier de sortie des resultats
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

# Connexion DuckDB
con = duckdb.connect()
pq = str(FICHIER)

# Verification
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
taille_mo = FICHIER.stat().st_size / (1024**2)
print(f"Fichier : {FICHIER.name}")
print(f"Taille sur disque : {taille_mo:.0f} Mo")
print(f"Dossier de sortie : {SORTIE}")

Fichier : dvf-2022.parquet
Taille sur disque : 105 Mo
Dossier de sortie : ./figures


## Cellule 3 — Combien de lignes et de colonnes ?

Premiere question a poser a tout jeu de donnees :
quelle est sa taille (nombre de lignes) et sa largeur (nombre de colonnes) ?

In [3]:
# Nombre total de lignes
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]

# Liste des colonnes et de leurs types
colonnes = con.execute(f"DESCRIBE SELECT * FROM '{pq}'").fetchall()
col_names = [c[0] for c in colonnes]

print(f"Nombre de lignes  : {nb_lignes:,}".replace(",", " "))
print(f"Nombre de colonnes : {len(colonnes)}")

Nombre de lignes  : 4 617 590
Nombre de colonnes : 43


## Cellule 4 — Quelles sont les colonnes et leurs types ?

Chaque colonne stocke un type de donnees particulier :

- `VARCHAR` = texte (noms, codes, identifiants alphanumeriques)
- `BIGINT` ou `INTEGER` = nombre entier (surfaces, compteurs)
- `DOUBLE` = nombre decimal (montants, coordonnees)
- `DATE` = date calendaire

Ce type conditionne les operations possibles : on peut calculer
une moyenne sur un nombre, pas sur du texte.

In [4]:
print(f"{'#':<4} {'Nom de la colonne':<35} {'Type'}")
print("-" * 60)
for i, (nom, typ, *_) in enumerate(colonnes, 1):
    print(f"{i:<4} {nom:<35} {typ}")

#    Nom de la colonne                   Type
------------------------------------------------------------
1    Identifiant de document             VARCHAR
2    Reference document                  VARCHAR
3    1 Articles CGI                      VARCHAR
4    2 Articles CGI                      VARCHAR
5    3 Articles CGI                      VARCHAR
6    4 Articles CGI                      VARCHAR
7    5 Articles CGI                      VARCHAR
8    No disposition                      VARCHAR
9    Date mutation                       DATE
10   Nature mutation                     VARCHAR
11   Valeur fonciere                     INTEGER
12   No voie                             BIGINT
13   B/T/Q                               VARCHAR
14   Type de voie                        VARCHAR
15   Code voie                           VARCHAR
16   Voie                                VARCHAR
17   Code postal                         VARCHAR
18   Commune                             VARCHAR
19   Code depar

## Cellule 5 — Apercu des premieres lignes

Avant toute analyse chiffree, regarder quelques lignes brutes
pour se faire une premiere idee du contenu. Observer en particulier :
les valeurs qui se repetent, les colonnes vides, les formats de date.

In [1]:
apercu = con.execute(f"""
    SELECT *
    FROM '{pq}'
    LIMIT 10
""").fetchdf()

display(apercu)

NameError: name 'con' is not defined

## Cellule 6 — Existe-t-il des colonnes entierement vides ?

Certains jeux de donnees publics conservent des colonnes dans leur
structure mais en suppriment le contenu (pour des raisons legales,
techniques ou de confidentialite).

Cette cellule identifie les colonnes dont **toutes les valeurs sont NULL**.
Ces colonnes ne pourront pas etre exploitees dans l'analyse.

In [5]:
cols_vides = []
cols_exploitables = []

for c in col_names:
    n_null = con.execute(f'SELECT count(*) FROM \'{pq}\' WHERE "{c}" IS NULL').fetchone()[0]
    if n_null == nb_lignes:
        cols_vides.append(c)
    else:
        cols_exploitables.append(c)

print("Colonnes 100 % vides :")
print("-" * 40)
if cols_vides:
    for c in cols_vides:
        print(f"  {c}")
else:
    print("  (aucune)")

print(f"\nTotal : {len(cols_vides)} colonnes vides")
print(f"Colonnes exploitables : {len(cols_exploitables)}")

Colonnes 100 % vides :
----------------------------------------
  Identifiant de document
  Reference document
  1 Articles CGI
  2 Articles CGI
  3 Articles CGI
  4 Articles CGI
  5 Articles CGI
  Identifiant local

Total : 8 colonnes vides
Colonnes exploitables : 35


## Cellule 7 — Taux de valeurs manquantes par colonne

Une **valeur manquante** (NULL) est une case vide dans le tableau.
Le taux de manquant indique, pour chaque colonne, quel pourcentage
de lignes n'a pas de valeur renseignee.

Un taux eleve n'est pas forcement un probleme : il peut refleter
la structure meme des donnees. L'interpretation viendra apres.

Les colonnes 100 % vides (deja identifiees) sont exclues de l'affichage.

In [6]:
print(f"{'Colonne':<35} {'Manquants':>12} {'Taux (%)':>10}")
print("-" * 60)

resultats_manquants = []
for c in col_names:
    n_null = con.execute(f'SELECT count(*) FROM \'{pq}\' WHERE "{c}" IS NULL').fetchone()[0]
    pct = round(100 * n_null / nb_lignes, 2)
    resultats_manquants.append((c, n_null, pct))

# Affichage trie par taux decroissant, hors colonnes 100 % vides et 0 %
for c, n, pct in sorted(resultats_manquants, key=lambda x: x[2], reverse=True):
    if 0 < pct < 100:
        print(f"  {c:<35} {n:>10,} {pct:>9.2f} %".replace(",", " "))

# Colonnes sans aucun manquant
completes = [c for c, n, pct in resultats_manquants if pct == 0 and c not in cols_vides]
print(f"\nColonnes 100 % completes ({len(completes)}) :")
for c in completes:
    print(f"  {c}")

Colonne                                Manquants   Taux (%)
------------------------------------------------------------
  Surface Carrez du 5eme lot           4 616 138     99.97 %
  Surface Carrez du 4eme lot           4 613 682     99.92 %
  No Volume                            4 607 192     99.77 %
  5eme lot                             4 606 249     99.75 %
  Surface Carrez du 3eme lot           4 601 600     99.65 %
  4eme lot                             4 591 146     99.43 %
  3eme lot                             4 535 192     98.22 %
  Surface Carrez du 2eme lot           4 475 948     96.93 %
  Nature culture speciale              4 428 905     95.91 %
  B/T/Q                                4 406 247     95.42 %
  Prefixe de section                   4 401 405     95.32 %
  Surface Carrez du 1er lot            4 198 313     90.92 %
  2eme lot                             4 159 584     90.08 %
  1er lot                              3 119 800     67.56 %
  Surface reelle bati    

## Cellule 8 — Statistiques descriptives de la valeur fonciere

La **valeur fonciere** correspond au montant de la transaction.
Les statistiques descriptives resument sa distribution :

- **Minimum / Maximum** : les bornes extremes.
- **Mediane** : la valeur qui separe le jeu en deux moities egales.
  Elle est plus representative que la moyenne quand la distribution
  est deformee par des valeurs extremes.
- **Moyenne** : la somme des valeurs divisee par le nombre de valeurs.
- **Percentiles** (P1, P5, Q1, Q3, P95, P99) : des marqueurs de position.
  Par exemple, P5 = 5 % des transactions sont en dessous de cette valeur.
  Q1 (= P25) et Q3 (= P75) delimitent les 50 % centraux.

Observer l'ecart entre moyenne et mediane : s'il est important,
la distribution est **asymetrique** (quelques valeurs extremes
tirent la moyenne vers le haut ou vers le bas).

In [7]:
stats = con.execute(f"""
    SELECT
        count(*)                                          AS nb_total,
        count("Valeur fonciere")                          AS nb_non_null,
        count(*) - count("Valeur fonciere")               AS nb_null,
        count(CASE WHEN "Valeur fonciere" = 0 THEN 1 END) AS nb_zero,
        min("Valeur fonciere")                            AS minimum,
        approx_quantile("Valeur fonciere", 0.01)          AS P1,
        approx_quantile("Valeur fonciere", 0.05)          AS P5,
        approx_quantile("Valeur fonciere", 0.25)          AS Q1,
        median("Valeur fonciere")                          AS mediane,
        approx_quantile("Valeur fonciere", 0.75)          AS Q3,
        approx_quantile("Valeur fonciere", 0.95)          AS P95,
        approx_quantile("Valeur fonciere", 0.99)          AS P99,
        max("Valeur fonciere")                            AS maximum,
        round(avg("Valeur fonciere"), 0)                  AS moyenne
    FROM '{pq}'
""").fetchone()

labels = ["Nb total", "Nb non null", "Nb null", "Nb a 0",
          "Minimum", "P1", "P5", "Q1", "Mediane", "Q3", "P95", "P99",
          "Maximum", "Moyenne"]

print("Statistiques descriptives — Valeur fonciere")
print("=" * 45)
for label, val in zip(labels, stats):
    if isinstance(val, (int, float)):
        print(f"  {label:<15} {val:>15,.0f}".replace(",", " "))
    else:
        print(f"  {label:<15} {val}")

Statistiques descriptives — Valeur fonciere
  Nb total              4 617 590
  Nb non null           4 586 448
  Nb null                  31 142
  Nb a 0                       33
  Minimum                       0
  P1                          182
  P5                        3 683
  Q1                       76 317
  Mediane                 175 000
  Q3                      321 507
  P95                   1 358 560
  P99                  48 144 473
  Maximum           1 003 401 470
  Moyenne               2 825 906


## Cellule 9 — Doublons exacts

Un **doublon exact** est une ligne dont toutes les valeurs sont
strictement identiques a celles d'une autre ligne.

La recherche porte sur les colonnes exploitables (celles qui ne sont
pas 100 % vides). Le resultat indique :

- Combien de **groupes** de lignes identiques existent.
- Combien de **lignes en trop** il y a (dans un groupe de 3 lignes
  identiques, 2 sont "en trop").

La presence de doublons n'est pas forcement une erreur de saisie :
d'autres explications sont possibles (structure du fichier, suppressions
de colonnes discriminantes, etc.). L'interpretation viendra apres.

In [8]:
cols_sql = ", ".join([f'"' + c + '"' for c in cols_exploitables])

dup = con.execute(f"""
    SELECT
        count(*) AS nb_groupes_dupliques,
        sum(n)   AS lignes_dans_groupes,
        sum(n-1) AS lignes_en_trop
    FROM (
        SELECT count(*) AS n
        FROM '{pq}'
        GROUP BY {cols_sql}
        HAVING count(*) > 1
    )
""").fetchone()

print("Doublons exacts (colonnes exploitables)")
print("=" * 45)
print(f"  Groupes de doublons     : {dup[0]:>10,}".replace(",", " "))
print(f"  Lignes dans ces groupes : {dup[1]:>10,}".replace(",", " "))
print(f"  Lignes en trop          : {dup[2]:>10,}".replace(",", " "))
print(f"  Taux de doublons        : {100*dup[2]/nb_lignes:.2f} %")
print(f"\nApres deduplication : {nb_lignes - dup[2]:,} lignes".replace(",", " "))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Doublons exacts (colonnes exploitables)
  Groupes de doublons     :    191 468
  Lignes dans ces groupes :    536 750
  Lignes en trop          :    345 282
  Taux de doublons        : 7.48 %

Apres deduplication : 4 272 308 lignes


## Cellule 10 — Cardinalites des variables categorielles

La **cardinalite** d'une variable est le nombre de valeurs distinctes
qu'elle contient. Par exemple, une colonne avec les valeurs
"A", "B", "C" a une cardinalite de 3.

Cette mesure permet de reperer :
- les variables a **faible cardinalite** (peu de categories) :
  candidates a un encodage categoriel pour un modele ML.
- les variables a **haute cardinalite** (beaucoup de valeurs differentes) :
  a traiter differemment (regroupement, suppression, etc.).

Les 3 valeurs les plus frequentes de chaque variable sont affichees
pour donner un apercu du contenu.

In [9]:
# Selection des colonnes textuelles ou a faible cardinalite attendue
cat_cols = ["Nature mutation", "Type local", "Code type local",
            "Nature culture", "Nature culture speciale",
            "Code departement", "Nombre de lots"]

print(f"{'Variable':<30} {'Modalites':>10}  Valeurs les plus frequentes")
print("-" * 90)

for c in cat_cols:
    card = con.execute(f'SELECT count(DISTINCT "{c}") FROM \'{pq}\'').fetchone()[0]
    top3 = con.execute(f"""
        SELECT "{c}", count(*) AS n
        FROM '{pq}'
        WHERE "{c}" IS NOT NULL
        GROUP BY 1 ORDER BY 2 DESC LIMIT 3
    """).fetchall()
    top_str = ", ".join([f"{v} ({n:,})".replace(",", " ") for v, n in top3])
    print(f"  {c:<30} {card:>8}  {top_str}")

Variable                        Modalites  Valeurs les plus frequentes
------------------------------------------------------------------------------------------
  Nature mutation                       6  Vente (4 267 222), Vente en l'état futur d'achèvement (280 574), Echange (45 200)
  Type local                            4  Dépendance (1 203 439), Maison (756 009), Appartement (638 879)
  Code type local                       4  3 (1 203 439), 1 (756 009), 2 (638 879)
  Nature culture                       27  S (1 575 389), T (421 587), P (225 299)
  Nature culture speciale             127  POTAG (43 386), PARC (19 620), PATUR (19 299)
  Code departement                     97  59 (122 665), 33 (119 757), 13 (108 534)
  Nombre de lots                       78  0 (3 119 800), 1 (1 039 784), 2 (375 608)


## Cellule 11 — Distribution des variables categorielles

Pour chaque variable categorielle a faible cardinalite,
afficher la distribution complete : toutes les valeurs possibles,
leur effectif et leur pourcentage.

Cela permet de voir si certaines categories dominent fortement,
si d'autres sont marginales, et si des valeurs inattendues existent.

In [10]:
# Distribution de Nature mutation
nat_mut = con.execute(f"""
    SELECT "Nature mutation",
           count(*) AS nb,
           round(100.0 * count(*) / {nb_lignes}, 2) AS pct
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

print("Distribution de Nature mutation")
display(nat_mut)

Distribution de Nature mutation


,Nature mutation,nb,pct
0,Vente,4267222,92.41
1,Vente en l'état futur d'achèvement,280574,6.08
2,Echange,45200,0.98
3,Vente terrain à bâtir,14268,0.31
4,Adjudication,9424,0.20
5,Expropriation,902,0.02


In [11]:
# Distribution de Type local
type_local = con.execute(f"""
    SELECT "Type local",
           count(*) AS nb,
           round(100.0 * count(*) / {nb_lignes}, 2) AS pct
    FROM '{pq}'
    WHERE "Type local" IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

print("Distribution de Type local (hors NULL)")
display(type_local)

Distribution de Type local (hors NULL)


,Type local,nb,pct
0,Dépendance,1203439,26.06
1,Maison,756009,16.37
2,Appartement,638879,13.84
3,Local industriel. commercial ou assimilé,142535,3.09


In [ ]:
# Distribution de Nature culture (top 15)
nat_cult = con.execute(f"""
    SELECT "Nature culture",
           count(*) AS nb,
           round(100.0 * count(*) / {nb_lignes}, 2) AS pct
    FROM '{pq}'
    WHERE "Nature culture" IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").fetchdf()

print("Top 15 des Nature culture")
display(nat_cult)

## Cellule 12 — Le manquant depend-il du type de bien ?

A la cellule 7, on a observe que certaines colonnes ont un taux de
manquant important (surfaces, type de local). La question naturelle
qui suit est : **ce manquant est-il reparti uniformement, ou depend-il
d'une autre variable ?**

Ce croisement examine le taux de manquant des colonnes `Surface reelle bati`
et `Surface terrain` en fonction du `Code type local`.
Si le taux varie fortement selon le type, le manquant n'est pas aleatoire.

In [12]:
mnar = con.execute(f"""
    SELECT type_local, nb_lignes, pct_bati_vide, pct_terrain_vide
    FROM (
        SELECT
            CASE
                WHEN "Code type local" IS NULL THEN '(vide)'
                WHEN "Code type local" = 1 THEN '1 - Maison'
                WHEN "Code type local" = 2 THEN '2 - Appartement'
                WHEN "Code type local" = 3 THEN '3 - Dependance'
                WHEN "Code type local" = 4 THEN '4 - Local ind./com.'
            END AS type_local,
            "Code type local" AS ctl_raw,
            count(*) AS nb_lignes,
            round(100.0 * sum(CASE WHEN "Surface reelle bati" IS NULL THEN 1 ELSE 0 END) / count(*), 1)
                AS pct_bati_vide,
            round(100.0 * sum(CASE WHEN "Surface terrain" IS NULL THEN 1 ELSE 0 END) / count(*), 1)
                AS pct_terrain_vide
        FROM '{pq}'
        GROUP BY "Code type local"
    )
    ORDER BY CASE WHEN ctl_raw IS NULL THEN 99 ELSE ctl_raw END
""").fetchdf()

print("Taux de manquant des surfaces, ventile par type de local")
print("=" * 65)
display(mnar)

Taux de manquant des surfaces, ventile par type de local


,type_local,nb_lignes,pct_bati_vide,pct_terrain_vide
0,1 - Maison,756009,0.0,4.3
1,2 - Appartement,638879,0.0,77.0
2,3 - Dependance,1203439,0.0,56.4
3,4 - Local ind./com.,142535,1.8,42.6
4,(vide),1876728,100.0,13.6


## Cellule 13 — Synthese de l'etape 1

Resume les indicateurs principaux decouverts dans ce notebook,
pour faciliter la transition vers l'etape suivante.

In [13]:
print("Synthese — Profilage DVF 2022, etape 1")
print("=" * 55)
print(f"  Fichier              : {FICHIER.name}")
print(f"  Taille               : {taille_mo:.0f} Mo")
print(f"  Lignes               : {nb_lignes:,}".replace(",", " "))
print(f"  Colonnes (total)     : {len(col_names)}")
print(f"  Colonnes vides       : {len(cols_vides)}")
print(f"  Colonnes exploitables: {len(cols_exploitables)}")
print(f"  Doublons exacts      : {dup[2]:,} ({100*dup[2]/nb_lignes:.2f} %)".replace(",", " "))
print(f"")

# Valeur fonciere
print("  Valeur fonciere :")
for label, val in zip(labels, stats):
    if isinstance(val, (int, float)):
        print(f"    {label:<15} {val:>15,.0f}".replace(",", " "))
    else:
        print(f"    {label:<15} {val}")

Synthese — Profilage DVF 2022, etape 1
  Fichier              : dvf-2022.parquet
  Taille               : 105 Mo
  Lignes               : 4 617 590
  Colonnes (total)     : 43
  Colonnes vides       : 8
  Colonnes exploitables: 35
  Doublons exacts      : 345 282 (7.48 %)

  Valeur fonciere :
    Nb total              4 617 590
    Nb non null           4 586 448
    Nb null                  31 142
    Nb a 0                       33
    Minimum                       0
    P1                          182
    P5                        3 683
    Q1                       76 317
    Mediane                 175 000
    Q3                      321 507
    P95                   1 358 560
    P99                  48 144 473
    Maximum           1 003 401 470
    Moyenne               2 825 906


## Cellule 14 — Fermeture

Fermer la connexion DuckDB pour liberer les ressources.

In [14]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
